In [5]:
import sys
import os
import pandas as pd
from pathlib import Path

In [6]:
# Project setup
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data" / "raw").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

- Show what columns exist and what a few rows look like
- Followed this as a rule of thumb: peek with pandas, process with Spark.

In [54]:
for name in ["air_quality", "weather", "taxi_zones"]:
    folder = Path("data/raw") / name
    files = sorted(folder.glob("*.csv"))
    print(f"--{name}: {[f.name for f in files]}--")

    df = pd.read_csv(files[0])
    print("Columns:", list(df.columns))
    print(df.head(3))
    print()

In [56]:
from src.common.spark_session import get_spark

spark = get_spark()

trips = spark.read.parquet("data/raw/taxi_trips")
print("Rows:", trips.count())
trips.printSchema()
trips.show(3, truncate=False)

### Checking for candiate Primary Keys

In [58]:
from pyspark.sql import functions as F

taxi_key = [
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID"
]

trips.groupBy(taxi_key).count() \
    .filter(F.col("count") > 1) \
    .show(10)

In [64]:
trips.groupBy(taxi_key).count() \
    .filter(F.col("count") > 1) \
    .count()

In [65]:
trips.filter(
    (F.col("VendorID") == 2) &
    (F.col("tpep_pickup_datetime") == "2024-01-01 00:04:00") &
    (F.col("tpep_dropoff_datetime") == "2024-01-01 00:04:44") &
    (F.col("PULocationID") == 63) &
    (F.col("DOLocationID") == 63)
).show(truncate=False)

- Primary key: No explicit primary key is provided in the dataset. The available attributes do not guarantee unique identification of individual trips.
- The Taxi Trips dataset contains multiple records sharing the same apparent trip-identifying attributes, so uniqueness cannot be assumed.

In [61]:
weather_df = pd.read_csv("data/raw/weather/weather.csv")
weather_key = ["year", "month", "day", "hour"]

duplicates = weather_df.duplicated(subset=weather_key).sum()

print("Duplicate records:", duplicates)
print("Nulls:")
print(weather_df[weather_key].isna().sum())

- Each combination identifies exactly one hourly weather observation.

In [62]:
air_df = pd.read_csv("data/raw/air_quality/hourly_88101_2024.csv")
air_key = [
    "State Code",
    "County Code",
    "Site Num",
    "Parameter Code",
    "POC",
    "Date Local",
    "Time Local"
]

duplicates = air_df.duplicated(subset=air_key).sum()

print("Duplicate records:", duplicates)
print("Nulls:")
print(air_df[air_key].isna().sum())

- The elements in the list `air key` can be treated as Primary Keys.

In [63]:
zones_df = pd.read_csv("data/raw/taxi_zones/taxi_zone_lookup.csv")
print("Duplicate LocationIDs:",zones_df["LocationID"].duplicated().sum())
print("Null LocationIDs:",zones_df["LocationID"].isna().sum())

### Checking for Categorical Column

In [73]:
for col in ["VendorID", "RatecodeID", "store_and_fwd_flag",
            "PULocationID", "DOLocationID", "payment_type"]:
    print(f"\n{col}")
    trips.select(col).distinct().show(20)

In [74]:
for col in [
    "temp_source",
    "rhum_source",
    "prcp_source",
    "snwd_source",
    "wdir_source",
    "wspd_source",
    "wpgt_source",
    "pres_source",
    "cldc_source",
    "coco"
]:
    print(f"\n{col}")
    print(weather_df[col].value_counts(dropna=False))

In [75]:
for col in [
    "Parameter Name",
    "Units of Measure",
    "Qualifier",
    "Method Type",
    "Method Code",
    "Method Name",
    "State Name",
    "County Name",
    "Datum"
]:
    print(f"\n--- {col} ---")
    print(air_df[col].value_counts(dropna=False).head(20))

In [76]:
for col in [
    "State Code",
    "County Code",
    "Site Num",
    "Parameter Code",
    "POC"
]:
    print(f"\n--- {col} ---")
    print("Unique:", air_df[col].nunique(dropna=False))
    print(air_df[col].value_counts(dropna=False).head(10))

## Task 2

In [53]:
from pathlib import Path

for sub in ["taxi_trips", "weather", "air_quality", "taxi_zones"]:
    files = sorted(Path("data/raw", sub).glob("*"))
    total = sum(f.stat().st_size for f in files)
    print(f"{sub:15s} {total/1e6:8.1f} MB  ({len(files)} file(s))")
    for f in files:
        print(f"    {f.name:45s} {f.stat().st_size/1e6:8.2f} MB")

## Task 5

In [3]:
pwd

In [7]:
from src.common.spark_session import get_spark

spark = get_spark()
df = spark.read.format("delta").load("data/silver/air_quality")
df.select("state_code", "county_code", "site_num", "state_name", "county_name", "latitude", "longitude") \
  .distinct() \
  .filter("state_name = 'New York'") \
  .show(20, truncate=False)
spark.stop()

In [9]:
from src.common.spark_session import get_spark
spark = get_spark()
df = spark.read.format("delta").load("data/silver/air_quality")
df = df.filter("state_code=36 AND county_code=81 AND site_num=124")
df.groupBy("date_local").count().filter("count > 1").show(5)
print("total rows for this site:", df.count())
spark.stop()

In [12]:
from src.common.spark_session import get_spark
spark = get_spark()
df = spark.read.format("delta").load("data/gold/integrated_taxi_trips")
print("gold rows:", df.count())
print("trip rows:", df.select("pickup_datetime","pickup_location_id","dropoff_location_id","total_amount").distinct().count())
spark.stop()

In [13]:
from src.common.spark_session import get_spark
from pyspark.sql.functions import col, lpad, concat, lit, to_timestamp

spark = get_spark()
df = spark.read.format("delta").load("data/silver/weather")
df = df.withColumn(
    "weather_ts",
    to_timestamp(
        concat(
            col("year").cast("string"), lit("-"),
            lpad(col("month").cast("string"), 2, "0"), lit("-"),
            lpad(col("day").cast("string"), 2, "0"), lit(" "),
            lpad(col("hour").cast("string"), 2, "0"), lit(":00:00")
        ),
        "yyyy-MM-dd HH:mm:ss"
    )
)
df.groupBy("weather_ts").count().filter("count > 1").show(5)
spark.stop()